# Data audit

Evidence for every number in the Data section of the [README](../README.md#data). Loading and cleaning live in `ds_challenge.data`.

In [1]:
import pandas as pd

from ds_challenge.data import (
    BIN_MINUTES,
    DURATION_BINS,
    LINES,
    Inc,
    Km,
    load_incidents,
    load_km,
    train_incidents,
)

inc = load_incidents()
km = load_km()
train = inc[inc[Inc.type] == "T"]
inc[Inc.type].value_counts()

type
T    29167
S    16394
Name: count, dtype: int64

In [2]:
# train incidents by line value: only a single named line can be put on that line's km
line = train[Inc.line]
kind = line.where(~line.isin(LINES), "single named line")
kind.mask(train[Inc.multi_line], "multi-line").value_counts()

line
single named line    28718
multi-line             351
Non affecté             97
02-avr                   1
Name: count, dtype: int64

## 1. Duration bin vs clock time (train incidents)

Agreement = clock minutes inside the bin, with 1 minute of tolerance.

In [3]:
bounds = pd.DataFrame.from_dict(BIN_MINUTES, orient="index", columns=["low", "high"])
b = train[Inc.duration_bin].astype(str)
agree = train[Inc.minutes].between(b.map(bounds["low"]) - 1, b.map(bounds["high"]) + 1)
long = train[Inc.minutes] > b.map(bounds["high"]) + 1
print("agreement:", round(agree.mean(), 4), "| disagreements:", (~agree).sum())
print("of which the clock runs longer than the bin:", long.sum())
pd.DataFrame(
    {
        "rows": b.value_counts(),
        "median_minutes": train.groupby(b)[Inc.minutes].median(),
        "disagree_share": (~agree).groupby(b).mean().round(3),
    }
).loc[DURATION_BINS]

agreement: 0.982 | disagreements: 525
of which the clock runs longer than the bin: 508


,rows,median_minutes,disagree_share
duration_bin,,,
02 min et moins,16450,0.0,0.010
03 à 04 min,6304,3.0,0.017
05 à 09 min,4377,6.0,0.039
10 à 14 min,933,11.0,0.055
15 à 19 min,299,17.0,0.047
20 à 24 min,173,22.0,0.052
25 à 29 min,107,27.0,0.065
30 min et plus,524,49.0,0.002


In [4]:
# share of train incidents where bin and clock disagree, by year
(~agree).groupby(train[Inc.year]).mean().round(4)

year
2019    0.0116
2020    0.0162
2021    0.0171
2022    0.0178
2023    0.0196
2024    0.0227
2025    0.0220
dtype: float64

## 2. Major-incident share by year (train incidents on a single named line, official bin)

In [5]:
train_incidents(inc).groupby(Inc.year).agg(
    incidents=(Inc.incident_id, "size"),
    major_5=(Inc.major_5, "mean"),
    major_10=(Inc.major_10, "mean"),
    major_10_n=(Inc.major_10, "sum"),
    major_30_n=(Inc.major_30, "sum"),
).round(3)

,incidents,major_5,major_10,major_10_n,major_30_n
year,,,,,
2019,5014,0.188,0.051,258,56
2020,3663,0.216,0.067,247,73
2021,3515,0.222,0.07,247,66
2022,4162,0.214,0.066,273,78
2023,4428,0.244,0.076,335,64
2024,4974,0.218,0.073,365,82
2025,2962,0.237,0.069,203,43


## 3. Flags against the 10+ outcome

Emergency team, power cut and evacuation track the outcome: they are set after the fact. Emergency brake and door barely differ.

In [6]:
flags = [Inc.emergency_brake, Inc.door, Inc.emergency_team, Inc.power_cut]
t = train.assign(evacuation=train[Inc.evacuation].notna())
t.groupby(Inc.major_10)[flags + [Inc.evacuation]].mean().T.round(3)

major_10,False,True
emergency_brake,0.035,0.057
door,0.101,0.089
emergency_team,0.005,0.172
power_cut,0.218,0.562
evacuation,0.019,0.419


## 4. Reporting lag: 2025 vs 2024 (train incidents per month)

In [7]:
monthly = train.groupby([train[Inc.year], train[Inc.date].dt.month]).size().unstack(0)
monthly.loc[1:9, [2023, 2024, 2025]]

year,2023,2024,2025
date,,,
1,449.0,490.0,511.0
2,365.0,427.0,427.0
3,425.0,424.0,439.0
4,357.0,367.0,311.0
5,313.0,381.0,288.0
6,333.0,345.0,247.0
7,380.0,391.0,268.0
8,327.0,502.0,299.0
9,360.0,535.0,257.0


## 5. Planned km

In [8]:
print("coverage:", km[Km.date].min().date(), "to", km[Km.date].max().date())
print(
    "unassigned rows:",
    km[Km.unassigned].sum(),
    "| share of km:",
    round(km.loc[km[Km.unassigned], Km.km].sum() / km[Km.km].sum(), 3),
)
km.groupby([Km.year, Km.unassigned])[Km.km].sum().unstack().div(1e6).round(1)

coverage: 2019-01-01 to 2023-09-30
unassigned rows: 1536 | share of km: 0.131


unassigned,False,True
year,,
2019,41.1,48.8
2020,83.5,4.5
2021,84.3,NaN
2022,84.4,NaN
2023,61.6,NaN
